In [ ]:
!pip install -q google-generativeai gradio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dataset_path = '/content/drive/MyDrive/Colab Notebooks/AI-A2/database-modul5-ai'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import google.generativeai as genai
import gradio as gr
import time
import getpass
import os

!pip install -q google-generativeai gradio chromadb sentence-transformers
import pickle
import chromadb
from sentence_transformers import SentenceTransformer

In [ ]:
# Inisialisasi model Gemini 2.5 Flash
model = genai.GenerativeModel("gemini-3.1-flash-lite")

print("✅ Gemini API berhasil dikonfigurasi!")
print("📌 Model: gemini-3.1-flash-lite")

✅ Gemini API berhasil dikonfigurasi!
📌 Model: gemini-3.1-flash-lite


In [ ]:
# CELL 3: Membangun Vector Database & Fungsi Retrieval Asli

# 1. Load data hasil ekstrak Orang 1 dari Drive
# Pastikan path ini sesuai dengan tempat file pkl kamu berada!
file_path = '/content/drive/MyDrive/Colab Notebooks/AI-A2/database-modul5-ai/all_chunks.pkl'

with open(file_path, 'rb') as f:
    all_chunks = pickle.load(f)

print(f"Berhasil memuat {len(all_chunks)} potongan dokumen dari Orang 1.")

# 2. Setup Model Pencarian & Database
print("Menyiapkan mesin pencari (Embedding & ChromaDB)...")
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
chroma_client = chromadb.Client()

# Buat database baru. Hapus jika sebelumnya sudah ada agar tidak duplikat saat di-run ulang
try:
    chroma_client.delete_collection(name="banaspati_docs")
except:
    pass
collection = chroma_client.create_collection(name="banaspati_docs")

# 3. Masukkan data ke Database (Ini memakan waktu sekitar 1-2 menit)
print("Memasukkan dokumen ke database... Mohon tunggu.")
texts = [c['text'] for c in all_chunks]
ids = [str(c['chunk_id']) for c in all_chunks]
metadatas = [{"source": c['source'], "page": c['page']} for c in all_chunks]

embeddings = embedding_model.encode(texts).tolist()
collection.add(documents=texts, embeddings=embeddings, metadatas=metadatas, ids=ids)
print("Selesai! Database siap digunakan.")

# 4. FUNGSI RETRIEVAL ASLI
def real_retrieve(query, top_k=3):
    # Cari dokumen paling mirip
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)

    # Format teks agar rapi saat masuk ke prompt dan UI
    konteks_teks = ""
    for i in range(len(results['documents'][0])):
        doc_text = results['documents'][0][i]
        doc_source = results['metadatas'][0][i]['source']
        doc_page = results['metadatas'][0][i]['page']

        # Format sesuai syarat evaluasi: menampilkan sumber
        konteks_teks += f"--- SUMBER {i+1}: {doc_source} (Halaman {doc_page}) ---\n{doc_text}\n\n"

    return konteks_teks



Berhasil memuat 1405 potongan dokumen dari Orang 1.
Menyiapkan mesin pencari (Embedding & ChromaDB)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Memasukkan dokumen ke database... Mohon tunggu.
Selesai! Database siap digunakan.


In [ ]:
# ============================================================
# SYSTEM PROMPT (Prompt Engineering)
# ============================================================

SYSTEM_PROMPT = """Kamu adalah BANASPATI, asisten cerdas Kedai Bubur Panas IT (ITS).
Tugasmu adalah menjawab pertanyaan seputar akademik dan operasional kedai.

ATURAN KETAT:
1. Jawab HANYA berdasarkan informasi pada KONTEKS yang diberikan.
2. Jika informasi untuk menjawab pertanyaan TIDAK ADA di dalam KONTEKS,
   kamu WAJIB menjawab: "Maaf, informasi tersebut tidak ditemukan dalam database kami.
   Silakan hubungi admin untuk informasi lebih lanjut."
3. JANGAN mengarang jawaban sendiri (jangan berhalusinasi).
4. Jawab dengan bahasa yang ramah, sopan, dan informatif.
5. Jika pertanyaan ambigu, minta klarifikasi dari user.
6. Gunakan format yang rapi (bullet points, numbering) jika jawaban panjang."""


# ============================================================
# PIPELINE RAG END-TO-END + TRACKING METRICS
# ============================================================

def generate_answer(pertanyaan):
    """
    Pipeline RAG end-to-end:
      1. Retrieve konteks (dari Orang 1 / mock)
      2. Inject konteks ke prompt (context injection)
      3. Generate jawaban dengan Gemini
      4. Track token usage & latency

    Returns:
      tuple: (konteks, jawaban, metrik_info)
    """
    try:
        waktu_mulai = time.time()

        # --- STEP 1: RETRIEVAL ---
        start_retrieval = time.time()
        konteks = real_retrieve(pertanyaan)
        retrieval_latency = time.time() - start_retrieval

        # --- STEP 2: CONTEXT INJECTION (Prompt Engineering) ---
        prompt = f"""{SYSTEM_PROMPT}

KONTEKS:
{konteks}

PERTANYAAN USER:
{pertanyaan}

JAWABAN:"""

        # --- STEP 3: GENERATION ---
        start_generation = time.time()
        response = model.generate_content(prompt)
        generation_latency = time.time() - start_generation

        # --- STEP 4: TRACKING TOKEN USAGE & LATENCY ---
        e2e_latency = time.time() - waktu_mulai

        try:
            input_tokens = response.usage_metadata.prompt_token_count
            output_tokens = response.usage_metadata.candidates_token_count
            total_tokens = response.usage_metadata.total_token_count
        except AttributeError:
            input_tokens = 0
            output_tokens = 0
            total_tokens = 0

        throughput = (
            output_tokens / generation_latency
            if generation_latency > 0
            else 0
        )

        # Estimasi Biaya (Gemini 2.5 Flash pricing)
        biaya_input = (input_tokens / 1_000_000) * 0.15
        biaya_output = (output_tokens / 1_000_000) * 0.60
        estimasi_biaya = biaya_input + biaya_output

        metrik_info = f"""### ⏱️ Latency
| Metrik | Nilai |
|--------|-------|
| Retrieval | {retrieval_latency:.4f}s |
| Generation | {generation_latency:.4f}s |
| End-to-End | {e2e_latency:.4f}s |

### 📊 Token Usage & Biaya
| Metrik | Nilai |
|--------|-------|
| Input Tokens | {input_tokens} |
| Output Tokens | {output_tokens} |
| Total Tokens | {total_tokens} |
| Throughput | {throughput:.2f} tok/sec |
| Est. Biaya | ${estimasi_biaya:.6f} |"""

        return konteks.strip(), response.text, metrik_info

    except Exception as e:
        return (
            "❌ Gagal mengambil konteks.",
            f"Terjadi kesalahan: {str(e)}",
            "❌ Metrik gagal dihitung."
        )


print("✅ Pipeline RAG end-to-end siap!")
print("📌 System prompt dikonfigurasi dengan fallback 'tidak ditemukan'")

✅ Pipeline RAG end-to-end siap!
📌 System prompt dikonfigurasi dengan fallback 'tidak ditemukan'


In [ ]:
# Quick test - tes pipeline tanpa Gradio
test_query = "Jam berapa kedai BANASPATI buka?"
konteks, jawaban, metrik = generate_answer(test_query)

print("=" * 60)
print("📝 Pertanyaan:", test_query)
print("=" * 60)
print()
print("📄 Konteks:")
print(konteks)
print()
print("💬 Jawaban:")
print(jawaban)
print()
print(metrik)

📝 Pertanyaan: Jam berapa kedai BANASPATI buka?

📄 Konteks:
--- SUMBER 1: Kalender-Akademik-ITS-Thn-Akademik-2025-2026.pdf (halaman 6) (Halaman 1) ---
Mohon maaf, berdasarkan gambar yang Anda lampirkan, **tidak terdapat jadwal perkuliahan** yang mencantumkan informasi Hari, Jam, Mata Kuliah, Kelas/Ruangan, atau Dosen.

Gambar tersebut merupakan **Kalender Akademik** yang berisi rincian jadwal administratif seperti:
*   Jadwal pendaftaran dan seleksi mahasiswa baru.
*   Jadwal registrasi administratif (pembayaran SPP).
*   Jadwal registrasi akademik (pengisian FRS).
*   Jadwal kegiatan perkuliahan (awal semester, masa evaluasi, pengisian nilai).
*   Jadwal yudisium dan wisuda.

Jika Anda memiliki dokumen lain yang berisi jadwal perkuliahan mingguan (hari dan jam kuliah), silakan unggah dokumen tersebut agar saya dapat membantu mengekstrak informasinya untuk Anda.

--- SUMBER 2: Sosialisasi Magang dan Prestasi DTI.pdf (Halaman 38) ---
Alur dan Timeline Institut Teknologi Sepuluh Nopember 

In [ ]:
# ============================================================
# DEMO SANDBOX - GRADIO CHATBOT UI
# ============================================================

custom_css = """
#header { text-align: center; margin-bottom: 10px; }
.gradio-container { max-width: 900px !important; margin: auto; }
"""

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="blue"),
    css=custom_css,
    title="BANASPATI - RAG Chatbot"
) as demo:

    gr.Markdown(
        """# 🍲 BANASPATI
### Asisten Cerdas Kedai Bubur Panas IT
---""",
        elem_id="header"
    )

    with gr.Row():
        with gr.Column(scale=3):
            input_teks = gr.Textbox(
                label="💬 Pertanyaan",
                placeholder="Contoh: Jam berapa kedai BANASPATI buka?",
                lines=2,
            )
            with gr.Row():
                tombol_tanya = gr.Button(
                    "🚀 Tanya BANASPATI",
                    variant="primary",
                    size="lg",
                )
                tombol_clear = gr.Button(
                    "🗑️ Clear Chat",
                    variant="secondary",
                    size="lg",
                )

    output_jawaban = gr.Chatbot(
        label="Percakapan dengan BANASPATI",
        height=350,
        type="messages",
        show_label=True,
    )

    with gr.Accordion("📋 Detail Konteks (Retrieval Debug)", open=False):
        output_konteks = gr.Textbox(
            label="Dokumen yang di-retrieve",
            lines=5,
            interactive=False,
        )

    output_metrik = gr.Markdown(
        value="*Metrik akan muncul setelah pertanyaan dijawab.*"
    )

    # --- Fungsi Chatbot ---
    def bot_response(query, history):
        if not query or not query.strip():
            return (
                history or [],
                "",
                "⚠️ Masukkan pertanyaan terlebih dahulu.",
            )
        konteks, jawaban, metrik = generate_answer(query)
        history = history or []
        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": jawaban})
        return history, konteks, metrik

    def clear_chat():
        return (
            [],
            "",
            "*Metrik akan muncul setelah pertanyaan dijawab.*",
            "",
        )

    # --- Event Handlers ---
    tombol_tanya.click(
        fn=bot_response,
        inputs=[input_teks, output_jawaban],
        outputs=[output_jawaban, output_konteks, output_metrik],
    )

    # Submit dengan Enter
    input_teks.submit(
        fn=bot_response,
        inputs=[input_teks, output_jawaban],
        outputs=[output_jawaban, output_konteks, output_metrik],
    )

    tombol_clear.click(
        fn=clear_chat,
        outputs=[output_jawaban, output_konteks, output_metrik, input_teks],
    )

# Jalankan demo
demo.launch(debug=True)

/tmp/ipykernel_918/2246243598.py:10: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_918/2246243598.py:10: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_918/2246243598.py:42: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  output_jawaban = gr.Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c9d39adfa0ccc1e255.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
